# Resources

> 

In [ ]:
#| default_exp resources

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional, Iterable, Tuple, Set

import io, os, re, sys, json
from pathlib import Path

from mcp.server.fastmcp import FastMCP  # official MCP Python SDK (FastMCP 2.0+)
from mcp.server.fastmcp.resources import FunctionResource

from nbdev_mcp.utils.config import load_bookmarks
from nbdev_mcp.utils.paths import (
    require_project,
    project_summary,
    nbs_dir,
    settings_dict,
    env_file,
    nbdev_settings_path,
    nbdev_generation,
)


## Resources

In [ ]:
#| export
def resource_project_summary() -> str:
    """Resource: JSON summary of the current project.
    
    Returns
    -------
    str
        JSON string with lib_name, nbs_path, notebooks count, and other project metadata.
    """
    p = require_project()
    return json.dumps(project_summary(p), indent=2)


In [ ]:
#| export
def resource_projects() -> str:
    """Resource: JSON of saved project bookmarks and NBDEV_PROJECTS env variable.
    
    Returns
    -------
    str
        JSON string with 'bookmarks' dict and 'NBDEV_PROJECTS' env value.
    """
    data = {'bookmarks': load_bookmarks(), 'NBDEV_PROJECTS': os.environ.get('NBDEV_PROJECTS', '')}
    return json.dumps(data, indent=2)


In [ ]:
#| export
def resource_tree() -> str:
    """Resource: JSON listing of notebooks in the current project.
    
    Returns
    -------
    str
        JSON string with project tree information and nbdev config metadata.
    """
    p = require_project()
    nbs = nbs_dir(p)
    files: List[str] = []
    if nbs.exists():
        for q in nbs.rglob('*.ipynb'):
            try:
                files.append(str(q.relative_to(p)))
            except Exception:
                files.append(str(q))

    settings_file = nbdev_settings_path(p)
    payload = {
        'root': str(p),
        'nbs_dir': str(nbs),
        'notebooks': files[:600],
        'has_settings_ini': (p / 'settings.ini').exists(),
        'has_settings_toml': (p / 'settings.toml').exists(),
        'settings_file': settings_file.name if settings_file is not None else None,
        'nbdev_generation': nbdev_generation(p),
        'has_readme': (p / 'README.md').exists(),
    }
    return json.dumps(payload, indent=2)


In [ ]:
#| export
def resource_settings() -> str:
    """Resource: contents of the current project's nbdev settings file.
    
    Returns
    -------
    str
        Contents of detected settings file (settings.ini/settings.toml/pyproject.toml),
        or an error message if not found.
    """
    p = require_project()
    settings_file = nbdev_settings_path(p)
    if settings_file is None:
        return 'No nbdev settings file found (expected settings.ini, settings.toml, or pyproject.toml with [tool.nbdev]).'
    return settings_file.read_text(encoding='utf-8')


In [ ]:
#| export
def resource_env_file() -> str:
    """Resource: contents of the current project's environment YAML file.
    
    Returns
    -------
    str
        The env YAML file contents (environment.yml or env.yml),
        or an error message if not found.
    """
    p = require_project()
    ef = env_file(p)
    return ef.read_text(encoding='utf-8') if ef.exists() else f'No env file at {ef}'


In [ ]:
#| export
def resource_read_file(relpath: str) -> str:
    """Resource: read a file by relative path within the current project.
    
    Parameters
    ----------
    relpath : str
        Relative path from project root to the file.
    
    Returns
    -------
    str
        The file contents, or an error message if not found or outside project.
    """
    p = require_project()
    f = (p / relpath).resolve()
    try:
        f.relative_to(p)
    except Exception:
        return f'Refusing to read outside project: {f}'
    if not f.exists():
        return f'Not found: {f}'
    try:
        return f.read_text(encoding='utf-8')
    except Exception as e:
        return f'Could not read {f}: {e}'


In [ ]:
#| export
def resource_roadmap() -> str:
    """Resource: pointer to roadmap.ipynb if it exists.
    
    Searches for roadmap.ipynb in the project root or nbs/ directory.
    
    Returns
    -------
    str
        JSON string with 'path' (relative path or None) and 'exists' flag.
    """
    p = require_project()
    candidates = [p / "roadmap.ipynb", nbs_dir(p) / "roadmap.ipynb"]
    for c in candidates:
        if c.exists():
            return json.dumps({"path": str(c.relative_to(p)), "exists": True}, indent=2)
    return json.dumps({"path": None, "exists": False, "message": "roadmap.ipynb not found in project root or nbs/"}, indent=2)


In [ ]:
#| export
def resource_index_to_readme_note() -> str:
    """Resource: explains that nbs/index.ipynb becomes README.md in nbdev.
    
    Returns
    -------
    str
        JSON string with 'lib' (library name) and 'message' explaining
        the index.ipynb to README.md relationship.
    """
    p = require_project()
    lib = settings_dict(p).get('lib_name') or '<your_lib>'
    msg = 'In nbdev, `nbs/index.ipynb` becomes `README.md` (the docs home page).'
    return json.dumps({'lib': lib, 'message': msg}, indent=2)


In [ ]:
#| export
def resource_learning_links() -> str:
    """Resource: curated links for nbdev, fastai, and related documentation.
    
    Returns
    -------
    str
        JSON string with categorized links to official documentation,
        tutorials, and community resources for nbdev, fastai, fastcore,
        execnb, quarto, and MCP protocol.
    """
    payload = {
        "nbdev": {
            "docs": "https://nbdev.fast.ai/",
            "getting_started": "https://nbdev.fast.ai/tutorials/tutorial.html",
            "best_practices": "https://nbdev.fast.ai/tutorials/best_practices.html",
            "directives": "https://nbdev.fast.ai/explanations/directives.html",
            "git_friendly": "https://nbdev.fast.ai/tutorials/git_friendly_jupyter.html",
            "github": "https://github.com/fastai/nbdev",
        },
        "fastai": {
            "docs": "https://docs.fast.ai/",
            "course": "https://course.fast.ai/",
            "github": "https://github.com/fastai/fastai",
            "forums": "https://forums.fast.ai/",
        },
        "fastcore": {
            "docs": "https://fastcore.fast.ai/",
            "github": "https://github.com/fastai/fastcore",
        },
        "execnb": {
            "docs": "https://execnb.fast.ai/",
            "github": "https://github.com/fastai/execnb",
        },
        "related": {
            "quarto": "https://quarto.org/",
            "jupyter": "https://jupyter.org/",
            "mcp_protocol": "https://modelcontextprotocol.io/",
        },
    }
    return json.dumps(payload, indent=2)


In [ ]:
#| export
def resource_version() -> str:
    """Resource: package version and Python version information.
    
    Returns
    -------
    str
        JSON string with 'nbdev_mcp' version, 'python' version,
        'platform', and 'dependencies' dict with nbdev, mcp, fastcore, rich.
    """
    import sys
    import platform
    
    # Get package version
    try:
        from importlib.metadata import version as pkg_version
        nbdev_mcp_version = pkg_version('nbdev-mcp')
    except Exception:
        nbdev_mcp_version = 'unknown'
    
    # Get dependency versions
    deps = {}
    for pkg in ['nbdev', 'mcp', 'fastcore', 'rich']:
        try:
            from importlib.metadata import version as pkg_version
            deps[pkg] = pkg_version(pkg)
        except Exception:
            deps[pkg] = 'not installed'
    
    payload = {
        'nbdev_mcp': nbdev_mcp_version,
        'python': sys.version.split()[0],
        'platform': platform.system(),
        'dependencies': deps
    }
    return json.dumps(payload, indent=2)


In [ ]:
#| export
import time

def resource_heartbeat() -> str:
    """Resource: MCP heartbeat for persistency monitoring.
    
    Returns current timestamp and uptime info. If this resource
    fails to respond, the MCP server is likely down and needs restart.
    
    Returns
    -------
    str
        JSON string with 'status', 'timestamp', 'timestamp_iso',
        'uptime_seconds', 'uptime_human', 'pid', and 'message'.
    """
    import os
    
    # Track server start time (approximate via module load time)
    if not hasattr(resource_heartbeat, '_start_time'):
        resource_heartbeat._start_time = time.time()
    
    uptime = time.time() - resource_heartbeat._start_time
    
    payload = {
        'status': 'alive',
        'timestamp': time.time(),
        'timestamp_iso': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        'uptime_seconds': int(uptime),
        'uptime_human': f'{int(uptime // 3600)}h {int((uptime % 3600) // 60)}m {int(uptime % 60)}s',
        'pid': os.getpid(),
        'message': 'nbdev-mcp server is running. If you cannot read this, restart the MCP server.'
    }
    return json.dumps(payload, indent=2)


In [ ]:
#| export
def add_resources(mcp: FastMCP) -> None:
    """Register all nbdev-mcp resources with the MCP server.
    
    Parameters
    ----------
    mcp : FastMCP
        The MCP server instance to register resources with.
    """
    mcp.add_resource(FunctionResource(
        uri="nbdev://project", name="project_summary",
        description="JSON summary of the current nbdev project",
        fn=resource_project_summary
    ))
    mcp.add_resource(FunctionResource(
        uri="nbdev://projects", name="projects",
        description="JSON of saved project bookmarks",
        fn=resource_projects
    ))
    mcp.add_resource(FunctionResource(
        uri="nbdev://tree", name="tree",
        description="JSON listing of notebooks in the project",
        fn=resource_tree
    ))
    mcp.add_resource(FunctionResource(
        uri="nbdev://roadmap", name="roadmap",
        description="Pointer to roadmap.ipynb if it exists",
        fn=resource_roadmap
    ))
    mcp.add_resource(FunctionResource(
        uri="nbdev://settings", name="settings",
        description="Contents of detected nbdev settings file",
        fn=resource_settings
    ))
    mcp.add_resource(FunctionResource(
        uri="nbdev://env", name="env_file",
        description="Contents of the environment YAML file",
        fn=resource_env_file
    ))
    mcp.add_resource(FunctionResource(
        uri="nbdev://index-readme", name="index_to_readme",
        description="Note about nbs/index.ipynb becoming README.md",
        fn=resource_index_to_readme_note
    ))
    mcp.add_resource(FunctionResource(
        uri="nbdev://links", name="learning_links",
        description="Curated links for nbdev/fastai documentation",
        fn=resource_learning_links
    ))
    mcp.add_resource(FunctionResource(
        uri="nbdev://version", name="version",
        description="Package version and dependency information",
        fn=resource_version
    ))
    mcp.add_resource(FunctionResource(
        uri="nbdev://heartbeat", name="heartbeat",
        description="MCP heartbeat for persistency monitoring - check this to verify MCP is alive",
        fn=resource_heartbeat
    ))


## Next

In [ ]:
#| export


## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()